# DC-02 Clean a page, then label it

**Time:** ~25 minutes &nbsp;|&nbsp; **Need:** a browser. No GPU. No extra packages.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/tier1/data_cleaning/DC-02_clean_then_label.ipynb)

Or **File → Upload notebook** at [colab.research.google.com](https://colab.research.google.com/).


**How to use this notebook**

1. **Runtime → Change runtime type → CPU** (not GPU), or run in the course Jupyter.
2. Click ▶ on each cell from the top, or **Runtime → Run all**.
3. Read the note *under* a cell after it runs.
4. Change **only** lines marked `# [CHANGE]`.
5. Nothing from work goes in a cell. CivicDesk names are fiction.


Cleaning here = make an *approved* paragraph searchable and honest. It is **not** pandas on a live extract, and it is **not** rewriting the law.


## What “clean” means on a government page

Keep:

- the circular number and the rule, word for word
- the office that issued it, the date, the clearance

Throw out:

- repeated letterheads and “page n of m”
- watermarks (`CONFIDENTIAL` copied onto a public handbook)
- OCR digits that were letters
- names, IDs, live ticket numbers

If you cannot get selectable text, **HOLD**. Do not run OCR on a real page in Colab.


## 1. Strip the soup — keep C2


In [ ]:
# [CHANGE] CLEAN must be the handbook sentence only. Do not "improve" the wording.
RAW = """CivicDesk Vacancy Handbook v3          page 4 of 88
C2. The app1icant must ho1d a recognised diploma in records management or equivalent.
CONFIDENTIAL — do not circulate            page 4 of 88
"""

CLEAN = "C2. The applicant must hold a recognised diploma in records management or equivalent."

print("RAW:")
print(RAW)
print("---")
print("CLEAN:")
print(CLEAN)

low = CLEAN.lower()
problems = []
if "page 4" in low or "confidential" in low:
    problems.append("header/footer still present")
if "app1icant" in low or "ho1d" in low:
    problems.append("OCR digit-for-letter still present")
if "records management" not in low:
    problems.append("the actual rule disappeared")
if "c2" not in low:
    problems.append("criterion id disappeared")
if "should probably" in low or "maybe" in low:
    problems.append("you rewrote the rule — that is not cleaning")
if problems:
    print("HOLD — still dirty:")
    for p in problems:
        print(" ", p)
else:
    print("Text is INDEX-ready. Next cell: labels that travel with it.")


### Teaching note

The gold line is already in the cell so you can see the target. The skill is *seeing* why RAW must not be indexed. A production desk re-types from the approved file or re-scans. Colab is not that desk.


## 2. A second page — overnight attachments (Finance-shaped)


In [ ]:
# [CHANGE] keep the rule; drop the email thread and the footer.
RAW = """From: A. Clerk  To: Whole floor
pls see attached, PS said this is the rule now??
---
CivicDesk Records Office  Circular HB-ATTACH-04   1 March 2024
Attachments may sit overnight only in the approved file store, not in personal email
and not in a public chatbot.
This email is confidential.  page 1 of 1
"""

CLEAN = (
    "Attachments may sit overnight only in the approved file store, "
    "not in personal email and not in a public chatbot."
)

print(CLEAN)
low = CLEAN.lower()
problems = []
if "pls see" in low or "ps said" in low:
    problems.append("email thread is not the circular")
if "page 1 of 1" in low or "this email is confidential" in low:
    problems.append("footer still present")
if "approved file store" not in low:
    problems.append("the rule disappeared")
if problems:
    print("HOLD:")
    for p in problems:
        print(" ", p)
else:
    print("Keep the circular. Throw the email. An inbox is not a handbook.")


## 3. Labels that travel with the paragraph

A chunk **missing** `id` or `clearance` is HOLD, even if the text is clean.

| Field | Why a government desk cares |
|---|---|
| `id` | So you can say later “this chunk was / was not in the index” |
| `title` | Human name of the circular |
| `source` | Handbook or circular number — not “the model told us” |
| `date` | Stale rules must not masquerade as current |
| `creator` | An **office**, never a name from a CV |
| `clearance` | internal / restricted / public — a hit is not permission |


In [ ]:
# [CHANGE] fill every quoted value. Fiction only.
CHUNK = {
    "id": "HB-VAC-C2",
    "title": "Vacancy Handbook v3 — criterion C2",
    "creator": "CivicDesk Records Office",
    "date": "2024-03-01",
    "source": "CivicDesk Vacancy Handbook v3",
    "clearance": "internal",
    "text": "C2. The applicant must hold a recognised diploma in records management or equivalent.",
}

required = ["id", "title", "source", "clearance", "text"]
missing = [k for k in required if not str(CHUNK.get(k, "")).strip()]
for k, v in CHUNK.items():
    print(f"{k:12} {v}")
print()
if missing:
    print("HOLD — empty:", ", ".join(missing))
elif not CHUNK["id"].startswith("HB-"):
    print("HOLD — id should look like HB-…")
elif CHUNK["clearance"] not in {"internal", "restricted", "public"}:
    print("HOLD — clearance is internal / restricted / public")
else:
    print("INDEX-ready. Day 2 stores these labels beside the paragraph, not inside the model.")
print("Creator is an office. A name from a CV would make this NEVER.")


## 4. Two versions of the same circular

If 2019 and 2026 both say “Circular 12” and neither is labelled **current**, HOLD both until Records says which one the desk uses. Do not let search retrieve the old one because it “looks similar.”


In [ ]:
# [CHANGE] current / stale / hold
GUESSES = {
    "HB-ATTACH-04, 1 March 2024, marked current by Records": "current",
    "HB-ATTACH-04, 12 June 2019, superseded note on the 2024 file": "stale",
    "Two undated photocopies both headed Circular 12": "hold",
}

gold = {
    "HB-ATTACH-04, 1 March 2024, marked current by Records": "current",
    "HB-ATTACH-04, 12 June 2019, superseded note on the 2024 file": "stale",
    "Two undated photocopies both headed Circular 12": "hold",
}
right = 0
for item, want in gold.items():
    got = GUESSES[item].strip().lower()
    ok = got == want
    right += int(ok)
    print(("OK   " if ok else "WRONG"), got, "—", item)
print(f"score {right}/{len(gold)}")
print("Stale circulars stay out of the pile, or sit in a clearly labelled archive that search does not use.")


## Done

You should be able to:

- strip letterhead and OCR junk **without rewriting** the rule
- refuse an email thread posing as policy
- fill `id`, `source`, `date`, `clearance`
- HOLD when two undated copies fight

Day 2 **T1-07** is the scored INDEX / NEVER drill. Day 2 **T1-10** is the access gate (a hit is not permission).
